In [2]:
import os
import sys
import spacy
import logging
import pandas as pd
from pandas import DataFrame
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForTokenClassification

d:\Document\NLP_Project\NER\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [5]:
def bert_person_extraction(model: object, tokenizer: object, text: str) -> str:
    
    nlp = pipeline("ner", model=model, tokenizer=tokenizer)
    ner_results = nlp(text)
    
    # Extract 'word' for entities of type 'PER'
    per_entities = [entity['word'] for entity in ner_results if entity['entity'] == 'B-PER' or entity['entity'] == 'I-PER']

    # save the extracted 'word' for PER entities
    person_name = " ".join(per_entities)
    
    return person_name
    

In [6]:
def spacy_info_extraction(prediction_data_df: DataFrame, spacy_model: object, bert_model: object,bert_tokenizer: object) -> dict:
    
    extracted_info = []
    for desc in prediction_data_df['Certificate Description']:
        doc = spacy_model(desc)

        sports_name = None
        winning_position = None
        org_year = None

        for ent in doc.ents:
            if ent.label_ == 'SPORTS_NAME':
                sports_name = ent.text
            elif ent.label_ == 'WINNING_POSITION':
                winning_position = ent.text
            elif ent.label_ == 'ORG_YEAR':
                org_year = ent.text

        # Extracting person name using the BERT model
        participant_name = bert_person_extraction(model=bert_model, tokenizer=bert_tokenizer, text=desc)

        extracted_info.append({
            'participent_name': participant_name,
            'sports_name': sports_name,
            'winning_position': winning_position,
            'org_year': org_year
        })

    return extracted_info
    

In [12]:


bert_tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")

bert_model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")

# Loading spaCy NER model
spacy_model = spacy.load("D:/Document/NLP_Project/NER/artifacts/ModelTrainerArtifacts/model-best")

prediction_data_df = pd.read_excel("D:/Document/NLP_Project/NER/data/data.xlsx", 'Data-Pending', header=None, names=['Certificate Description'])

# Extract information from the certificate descriptions
extracted_info = spacy_info_extraction(prediction_data_df=prediction_data_df, spacy_model=spacy_model,bert_model=bert_model, bert_tokenizer=bert_tokenizer)

# Create a new DataFrame with the extracted information
result_df = pd.DataFrame(extracted_info)


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device set to use cpu
Device s

In [13]:
print(result_df)

          participent_name        sports_name winning_position org_year
0             Erwin Harper          Athletics              1st     2010
1     El ##dridge Holloway            Surfing              2nd     2013
2    W ##il ##bert Bridges       Canoe Slalom              3rd     2010
3          Frederick Johns             Diving              3rd     2013
4             Kirby Rhodes         Water Polo              2nd     2009
..                     ...                ...              ...      ...
447       Frederick Miller               Judo              2nd     None
448       Clifford Jackson             Boxing              3rd     2013
449   Bert ##ram Lancaster          Taekwondo              3rd     2013
450            Harvey Gray        BMX Cycling              1st     None
451            Walter Hall  Modern Pentathlon              1st     2009

[452 rows x 4 columns]


In [14]:
print(prediction_data_df.head())

                             Certificate Description
0  In recognition of their exceptional performanc...
1  Eldridge Holloway, your outstanding performanc...
2  In recognition of their exceptional performanc...
3  Frederick Johns, your outstanding performance ...
4  In recognition of their exceptional performanc...
